In [ ]:
# -*- coding: utf-8 -*-
"""
PCA - Versión para Google Colab con Plotly
"""

import numpy as np
import pandas as pd
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
import plotly.express as px
import plotly.graph_objects as go
import os
import joblib
from google.colab import files  # Para descargar archivos fácilmente

In [ ]:
'''
###############################################################################
########## Leer archivo e indicar columnas que no se usarán #########
###############################################################################
'''

# Sube tu archivo CSV aquí
print("Por favor, sube tu archivo CSV:")
uploaded = files.upload()

# Obtener el nombre del archivo subido
filename = list(uploaded.keys())[0]
df = pd.read_csv(filename)

print(df.columns)
print(df.head())

In [ ]:
'''############################################################################
#################### Columnas indeseables si existen ######################
############################################################################'''
cols_not = []   # Escribe aquí las columnas que NO quieres usar

cols_use = [x for x in df.columns if x not in cols_not]
X = df[cols_use]
print("Columnas usadas:", X.columns.tolist())

In [ ]:
'''############################################################################
############### Definir columna objetivo si es que la hay ################
############################################################################'''
objetivo = []   # Si tienes target: objetivo = df["nombre_columna"]


In [ ]:
'''############################################################################
############### Escalado y cálculo de todas las componentes ###############
############################################################################'''

scaler = StandardScaler()
Z = scaler.fit_transform(X)

# PCA completo
pca_full = PCA()
pca_full.fit(Z)

varianza_acum = np.cumsum(pca_full.explained_variance_ratio_)

print(pd.DataFrame({
    "n_componentes": range(1, Z.shape[1] + 1),
    "varianza_acumulada": varianza_acum
}))

In [ ]:
'''############################################################################
################## Elección de la varianza mínima ######################
############################################################################'''

min_varianza = 0.95

k_varianza = np.argmax(varianza_acum >= min_varianza) + 1
diff = np.diff(varianza_acum)
k_codo = np.argmax(diff < 0.01) + 1
k_recomendado = min(k_varianza, k_codo)

print(f"k por mínima varianza ({min_varianza}): {k_varianza}")
print(f"k por criterio de codo: {k_codo} (varianza explicada: {varianza_acum[k_codo-1]:.4f})")
print(f"k recomendado: {k_recomendado}")

In [ ]:
'''############################################################################
################ Elección del número de componentes #####################
############################################################################'''

n_componentes = k_recomendado   # Puedes cambiar manualmente si quieres

pca = PCA(n_components=n_componentes)
Yk = pca.fit_transform(Z)

print(f"\nVarianza explicada con {n_componentes} componentes: {pca.explained_variance_ratio_.sum():.2%}")
print(f"Forma de Yk: {Yk.shape}")

In [ ]:
'''############################################################################
###################### GRÁFICAS CON PLOTLY ############################
############################################################################'''

if n_componentes == 1:
    print("\n⚠️ Solo una componente principal. No se puede graficar en 2D/3D.")

elif n_componentes == 2:
    print("\n📊 Graficando las 2 componentes principales con Plotly...")

    # Crear DataFrame para Plotly
    df_plot = pd.DataFrame(Yk, columns=['PC1', 'PC2'])
    if len(objetivo) > 0:
        df_plot['Target'] = objetivo.reset_index(drop=True) if isinstance(objetivo, pd.Series) else objetivo

    fig = px.scatter(
        df_plot, x='PC1', y='PC2',
        color='Target' if len(objetivo) > 0 else None,
        title=f'PCA - 2 Componentes Principales<br>Varianza explicada: {pca.explained_variance_ratio_.sum():.2%}',
        labels={'PC1': 'Componente Principal 1', 'PC2': 'Componente Principal 2'},
        opacity=0.8,
        width=900,
        height=700
    )
    fig.update_traces(marker=dict(size=10, line=dict(width=1, color='DarkSlateGrey')))
    fig.update_layout(template="plotly_white", title_x=0.5)
    fig.show()

else:  # n_componentes >= 3
    print(f"\n📊 Se usarán {n_componentes} componentes. Elige dimensión para graficar:")

    while True:
        respuesta = input("¿Quieres graficar en 2D o en 3D? (escribe 2 o 3): ").strip()
        if respuesta == '2':
            dim = 2
            break
        elif respuesta == '3':
            dim = 3
            break
        else:
            print("⚠️ Por favor escribe solo '2' o '3'")

    # Crear DataFrame
    cols = [f'PC{i+1}' for i in range(n_componentes)]
    df_plot = pd.DataFrame(Yk[:, :3], columns=cols[:3])  # tomamos las primeras 3 para graficar
    if len(objetivo) > 0:
        df_plot['Target'] = objetivo.reset_index(drop=True) if isinstance(objetivo, pd.Series) else objetivo

    if dim == 2:
        fig = px.scatter(
            df_plot, x='PC1', y='PC2',
            color='Target' if len(objetivo) > 0 else None,
            title=f'PCA - Primeras 2 Componentes<br>Varianza explicada: {pca.explained_variance_ratio_.sum():.2%}',
            opacity=0.8,
            width=900,
            height=700
        )
        fig.update_traces(marker=dict(size=10, line=dict(width=1, color='DarkSlateGrey')))
        fig.update_layout(template="plotly_white", title_x=0.5)
        fig.show()

    elif dim == 3:
        fig = px.scatter_3d(
            df_plot, x='PC1', y='PC2', z='PC3',
            color='Target' if len(objetivo) > 0 else None,
            title=f'PCA - Primeras 3 Componentes<br>Varianza explicada: {pca.explained_variance_ratio_.sum():.2%}',
            opacity=0.7,
            width=900,
            height=700
        )
        fig.update_traces(marker=dict(size=6, line=dict(width=1, color='DarkSlateGrey')))
        fig.update_layout(template="plotly_white", title_x=0.5)
        fig.show()

In [ ]:
'''############################################################################
################ GUARDAR Y DESCARGAR ARTEFACTOS ############################
############################################################################'''

import joblib
import numpy as np
from google.colab import files

# Guardar los archivos
joblib.dump(scaler, "pca_scaler.joblib")
joblib.dump(pca, "pca_model.joblib")
np.save("pca_cols_use.npy", np.array(cols_use))
np.save("Yk.npy", Yk)

Yk_tabla = pd.DataFrame(Yk, columns=[f"PC{i+1}" for i in range(n_componentes)])
Yk_tabla.to_csv("Yk.csv", index=False)

if len(objetivo) > 0:
    Yk_tabla["objetivo"] = objetivo
    Yk_tabla.to_csv("Yk_objetivo.csv", index=False)
    print("✓ Se guardó también el archivo con la variable objetivo")

# Descarga automática de todos los archivos
print("\n📥 Iniciando descarga automática de los archivos...")

files.download("pca_scaler.joblib")
files.download("pca_model.joblib")
files.download("pca_cols_use.npy")
files.download("Yk.npy")
files.download("Yk.csv")

if len(objetivo) > 0:
    files.download("Yk_objetivo.csv")

print("\n✅ ¡Todos los artefactos se han guardado y descargado automáticamente!")
print("\nArchivos descargados:")
print("   - pca_scaler.joblib")
print("   - pca_model.joblib")
print("   - pca_cols_use.npy")
print("   - Yk.npy")
print("   - Yk.csv")
if len(objetivo) > 0:
    print("   - Yk_objetivo.csv")

print("\n🎉 ¡Proceso de PCA terminado correctamente!")